# Step 1 — Exploration
getting a feel for the data before building anything

In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print(f"Train: {train.shape} | Test: {test.shape}")

# what columns does test NOT have? those are what we predict
print("Missing from test:", [c for c in train.columns if c not in test.columns])

In [ ]:
# quick sanity check
train.head(3)

In [ ]:
train.dtypes

In [ ]:
# face_emotion_hint has a lot missing — 10% gone, will need special handling
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)
pd.DataFrame({'count': missing, '%': missing_pct})[missing > 0]

In [ ]:
# are the targets balanced? matters a lot for model choice
print(train['emotional_state'].value_counts())
print()
print(train['intensity'].value_counts().sort_index())
# both look pretty even — no class weighting needed

In [ ]:
# categorical distributions — just eyeballing for now
for col in ['ambience_type', 'time_of_day', 'previous_day_mood',
            'face_emotion_hint', 'reflection_quality']:
    print(f"\n{col}:")
    print(train[col].value_counts(dropna=False))

In [ ]:
train[['duration_min', 'sleep_hours', 'energy_level', 'stress_level']].describe().round(2)

In [ ]:
# word count is more useful than char length for short noisy texts
train['word_count'] = train['journal_text'].str.split().str.len()
print(train['word_count'].describe())

# how many entries are dangerously short?
short = train[train['word_count'] <= 5]
print(f"\nShort texts (<=5 words): {len(short)}")
short[['journal_text', 'emotional_state', 'intensity']].head(6)

In [ ]:
# does stress actually track with emotional_state? checking my assumption
print(train.groupby('emotional_state')['stress_level'].mean().round(2).sort_values(ascending=False))
print()
print(train.groupby('emotional_state')['energy_level'].mean().round(2).sort_values(ascending=False))
# focused has high energy but low stress — makes sense
# overwhelmed has high stress but avg energy — interesting

In [ ]:
# curious: conflicted reflections — are they harder to classify?
# if yes, they might be noisy training samples
conflicted = train[train['reflection_quality'] == 'conflicted']
print(f"Conflicted rows: {len(conflicted)}")
print(conflicted['emotional_state'].value_counts())
print(f"avg stress in conflicted: {conflicted['stress_level'].mean():.2f}")
print(f"avg stress overall:       {train['stress_level'].mean():.2f}")

In [ ]:
# another hunch: when face_emotion_hint is missing, what's the emotional state?
# if it's random, we can just impute. if it's systematic, we need a flag feature
missing_face = train[train['face_emotion_hint'].isnull()]
print(f"Rows with missing face hint: {len(missing_face)}")
print(missing_face['emotional_state'].value_counts())
print()
# compare to overall distribution — is any state overrepresented?
print("Overall distribution:")
print(train['emotional_state'].value_counts())

In [ ]:
# spot check — just reading some rows to see if the data makes intuitive sense
pd.set_option('display.max_colwidth', 80)
train[['journal_text', 'emotional_state', 'intensity',
       'stress_level', 'energy_level', 'word_count']].sample(8, random_state=7)